# 预备知识：Tensor 与自动求导

## 本节目标
- 创建 Tensor、形状变换、广播、索引
- requires_grad / backward / grad
- 常见坑：inplace 操作、计算图断开

## 环境检查


In [1]:
import torch
print("torch:", torch.__version__)
print("mps:", torch.backends.mps.is_available())
print("cuda:", torch.cuda.is_available())


torch: 2.14.0
mps: True
cuda: False


## 实验区

> 在下方按课程步骤补全代码。


In [6]:
# TODO: start here
t1 = torch.tensor([10,20,30])
print("t1:", t1)
print("t1.shape:", t1.shape)
print("t1.dtype:", t1.dtype)


t1: tensor([10, 20, 30])
t1.shape: torch.Size([3])
t1.dtype: torch.int64


In [5]:
t1 = torch.tensor([[10,20,30],[20,30,40],[30,40,50]])
print("t1:", t1)
print("t1:", t1.shape)

t1: tensor([[10, 20, 30],
        [20, 30, 40],
        [30, 40, 50]])
t1: torch.Size([3, 3])


In [10]:
# 基于形状创建tensor
t1 = torch.Tensor(4,5)
print("t1:", t1)
print("t1.shape:", t1.shape)
print("t1.dtype:", t1.dtype)

t1: tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])
t1.shape: torch.Size([4, 5])
t1.dtype: torch.float32


In [11]:
# 创建制定类型的张量
t1 = torch.HalfTensor(4,5)
print("t1:", t1)
print("t1.shape:", t1.shape)
print("t1.dtype:", t1.dtype)

t1: tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]], dtype=torch.float16)
t1.shape: torch.Size([4, 5])
t1.dtype: torch.float16


In [12]:
t1 = torch.BoolTensor(4,5)
print("t1:", t1)
print("t1.shape:", t1.shape)
print("t1.dtype:", t1.dtype)

t1: tensor([[False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False]])
t1.shape: torch.Size([4, 5])
t1.dtype: torch.bool


In [17]:
# 制定区间创建张量
t1 = torch.arange(10)
print("t1:", t1)

t2 = torch.linspace(10,20, 3)
print("t2:", t2)


t1: tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
t2: tensor([10., 15., 20.])
t2.shape: torch.Size([3])


### 5、根据填充创建张量

In [18]:
t1 = torch.ones(3,4)
print("t1:", t1)

t1: tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])


In [19]:
t1 = torch.zeros(3,4)
print("t1:", t1)

t1: tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])


In [20]:
t1 = torch.empty(3,4)
print("t1:", t1)

t1: tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])


In [21]:
t1 = torch.full( (2,3), 6)
print("t1:", t1)

t1: tensor([[6, 6, 6],
        [6, 6, 6]])


In [22]:
t1 = torch.eye(6)
print("t1:", t1)

t1: tensor([[1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 1.]])


### 6、 随机创建张量

In [23]:
t1 = torch.rand(2,3) # 均匀分布 【0， 1】
print("t1:", t1)

t1: tensor([[0.8191, 0.1631, 0.2028],
        [0.4969, 0.5273, 0.8147]])


In [27]:
t1 = torch.randint(low=10, high=100, size=(2,3)) # 均匀分布 【0， 1】
print("t1:", t1)

t1: tensor([[59, 26, 74],
        [37, 72, 19]])


In [30]:
t1 = torch.randn(2,3) # 均匀分布 【0， 1】
print("t1:", t1)

t1: tensor([[-0.6315,  0.0346, -0.1102],
        [-2.8875,  0.5589,  1.5106]])


In [32]:
t1 = torch.normal(mean=5, std=3, size=(2,3)) # 均匀分布 【0， 1】
print("t1:", t1)

t1: tensor([[10.5570,  6.3248, -0.0136],
        [ 8.9426,  9.7969,  0.7755]])


### 1、 元素类型的转换

In [33]:
t1 = torch.tensor([1,2,3])
print("t1:", t1)
print("t1.shape:", t1.shape)

t1: tensor([1, 2, 3])
t1.shape: torch.Size([3])


### tensor --> ndarray

In [34]:
import numpy as  np

In [37]:
n1 = t1.numpy()
print("n1:", type(n1))
print("n1:", n1)

n1: <class 'numpy.ndarray'>
n1: [1 2 3]


In [38]:
t1[0] = 10
print("t1:", t1)
print("n1:", n1)


t1: tensor([10,  2,  3])
n1: [10  2  3]


In [40]:
n2 = t1.numpy().copy()
print("n1:", n2)
n2[0] = 100
print("n2:", n2)
print("n1:", n1)

n1: [10  2  3]
n2: [100   2   3]
n1: [10  2  3]


#### ndarray -> numpy

#### 张量 -> 标量

In [42]:
t1 = torch.tensor([[[10]]]) # tensor 只有一个元素的时候， 可以转
print("t1:", t1.item())

t1: 10


#### 矩阵相乘

In [48]:
t1 = torch.tensor([[10,20,30],[20,30,40]])
t2 = torch.tensor([[2,4],[5,7],[7,9]])
print("t1.shape:", t1.shape)
print("t2.shape:", t2.shape)
print("t1 @ t2:", t1 @ t2)

t1.shape: torch.Size([2, 3])
t2.shape: torch.Size([3, 2])
t1 @ t2: tensor([[330, 450],
        [470, 650]])


In [46]:
print("t1.mm(t2):", t1.mm(t2)) # 超过2维不支持


t1.mm(t2): tensor([[330, 390, 450],
        [470, 560, 650]])


In [47]:
print("t1.matmul(t2):", t1.matmul(t2))


t1.matmul(t2): tensor([[330, 390, 450],
        [470, 560, 650]])


#### 统计运算

In [57]:
t1 = torch.randint(1,10,(2,3,4))
print("t1:", t1)
print(t1.sum())


t1: tensor([[[8, 5, 2, 1],
         [8, 8, 7, 7],
         [4, 9, 9, 6]],

        [[7, 2, 3, 3],
         [5, 5, 5, 2],
         [8, 4, 6, 2]]])
tensor(126)


In [58]:
print(t1.unique())
# 去重后只保留一维

tensor([1, 2, 3, 4, 5, 6, 7, 8, 9])


#### 列表索引

In [61]:
list1 = [0,1,1]
list2 = [1,3,2]

t = torch.randint(0,10,(2,5,4))
print(t[list1,list2])

tensor([[5, 6, 3, 9],
        [5, 0, 1, 5],
        [5, 0, 7, 5]])


#### 布尔索引

In [65]:
mask = t > 5
mask


tensor([[[ True, False,  True, False],
         [False,  True, False,  True],
         [False, False, False, False],
         [ True,  True,  True,  True],
         [ True, False, False,  True]],

        [[ True,  True, False, False],
         [ True, False, False, False],
         [False, False,  True, False],
         [False, False, False, False],
         [ True,  True,  True,  True]]])

#### 维度变换

In [69]:
print(t)
print('---------------')
print(t.mT)

tensor([[[8, 2, 9, 2],
         [5, 6, 3, 9],
         [1, 4, 5, 1],
         [9, 6, 8, 8],
         [8, 4, 1, 7]],

        [[9, 9, 1, 4],
         [7, 3, 0, 5],
         [5, 0, 7, 5],
         [5, 0, 1, 5],
         [8, 9, 9, 7]]])
---------------
tensor([[[8, 5, 1, 9, 8],
         [2, 6, 4, 6, 4],
         [9, 3, 5, 8, 1],
         [2, 9, 1, 8, 7]],

        [[9, 7, 5, 5, 8],
         [9, 3, 0, 0, 9],
         [1, 0, 7, 1, 9],
         [4, 5, 5, 5, 7]]])
